# Convert the SignFlow ASL ResNet-50 model to ONNX

This notebook exports the MIT-licensed `Abuzaid01/asl-sign-language-classifier` model from PyTorch to ONNX for browser inference. It exports the model and the 29 output labels. This does **not** deploy the weights to SignFlow automatically; the resulting `model.onnx` must be hosted at a public HTTPS URL before the website can load it.

Model classes: A–Z, `del`, `nothing`, and `space`. A single image classifier does not reliably recognize motion-based J and Z in real time.

In [ ]:
!pip -q install --upgrade transformers torch torchvision onnx onnxruntime safetensors pillow

In [ ]:
import json, os, torch, onnx, onnxruntime
from pathlib import Path
from transformers import AutoImageProcessor, ASLResNet

MODEL_ID = 'Abuzaid01/asl-sign-language-classifier'
OUT = Path('/content/signflow_resnet50_onnx')
OUT.mkdir(parents=True, exist_ok=True)

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = ASLResNet.from_pretrained(MODEL_ID)
model.eval()
print('Loaded model:', MODEL_ID)
print('Input size:', processor.size)
print('Classes:', model.config.id2label)

In [ ]:
# Export a fixed 224x224 RGB image input. Dynamic batch size allows one or more images.
dummy = torch.zeros(1, 3, 224, 224, dtype=torch.float32)
onnx_path = OUT / 'model.onnx'
with torch.inference_mode():
    torch.onnx.export(
        model,
        (dummy,),
        str(onnx_path),
        input_names=['images'],
        output_names=['logits'],
        dynamic_axes={'images': {0: 'batch'}, 'logits': {0: 'batch'}},
        opset_version=17,
        do_constant_folding=True,
        dynamo=False,
    )

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
print('Exported:', onnx_path, f'({onnx_path.stat().st_size / 1024 / 1024:.1f} MiB)')

In [ ]:
labels_map = model.config.id2label
labels = [labels_map[str(i)] if str(i) in labels_map else labels_map[i] for i in range(len(labels_map))]
(OUT / 'labels.json').write_text(json.dumps(labels, indent=2))
preprocessing = {
    'input_size': 224,
    'image_mean': list(processor.image_mean),
    'image_std': list(processor.image_std),
    'rescale_factor': float(getattr(processor, 'rescale_factor', 1.0 / 255.0)),
    'resize': True,
    'color_mode': 'RGB',
    'layout': 'NCHW'
}
(OUT / 'preprocessing.json').write_text(json.dumps(preprocessing, indent=2))
print('Labels:', labels)
print('Preprocessing:', preprocessing)

In [ ]:
# Package the converted model, labels and preprocessing settings for hosting.
import zipfile
zip_path = Path('/content/signflow_resnet50_onnx.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(onnx_path, 'model.onnx')
    z.write(OUT / 'labels.json', 'labels.json')
    z.write(OUT / 'preprocessing.json', 'preprocessing.json')
print('Ready to download:', zip_path, f'({zip_path.stat().st_size / 1024 / 1024:.1f} MiB)')
from google.colab import files
files.download(str(zip_path))

## Next: host the model

Upload `model.onnx` and `labels.json` from the ZIP to a public model repository (for example, a new Hugging Face model repository you control). Then update SignFlow's ONNX model URL and preprocessing settings to that hosted location. Do not put the ZIP itself in the website's HTML. Keep the model repository public and ensure the model files allow browser CORS requests.